# Chunking Strategies — 20-Minute Live Demo

**Unit 3 companion demo — Module 1: Ingestion & Chunking Strategy**

This notebook is built to run live in ~20 minutes, cell by cell, followed by a set of
**extension tasks** for participants to try on their own afterward.

| Time | Section |
|---|---|
| 0:00–2:00 | Setup + the sample document |
| 2:00–5:00 | Strategy 1 — Fixed-Size Chunking |
| 5:00–8:00 | Strategy 2 — Sentence-Based Chunking |
| 8:00–11:00 | Strategy 3 — Recursive Character Chunking (with overlap) |
| 11:00–14:00 | Strategy 4 — Structure-Aware Chunking (by heading) |
| 14:00–17:00 | Strategy 5 — Semantic Chunking (embedding similarity) |
| 17:00–19:00 | Side-by-Side Comparison |
| 19:00–20:00 | Wrap-up + hand off to Extension Tasks |

> **Facilitator tip:** run each cell live and pause on the printed chunk boundaries —
> the goal is for participants to *see* each strategy fail or succeed on the same
> document, not just read about it.


In [ ]:
# Setup (0:00–1:00)
!pip install -q langchain-text-splitters sentence-transformers tiktoken


## The Sample Document (1:00–2:00)

A short policy document with a deliberately awkward structure: short clauses, a table-like
list, and section headings — enough to make naive chunking visibly break.


In [ ]:
sample_document = """# Auto Insurance Policy — Section 4: Coverage Details

## 4.1 Collision Coverage
The Company will pay for direct and accidental physical loss to your covered auto caused by
collision, subject to a $500 deductible per occurrence. This coverage applies regardless of
fault. Collision coverage does not apply to mechanical breakdown or normal wear and tear.

## 4.2 Comprehensive Coverage
The Company will pay for direct and accidental physical loss to your covered auto not caused
by collision, including but not limited to fire, theft, vandalism, and falling objects. A
$250 deductible applies per occurrence.

## 4.3 Rental Reimbursement
If your covered auto is out of service due to a covered collision or comprehensive loss, the
Company will reimburse rental costs up to $40 per day for a maximum of 30 days. Rental
reimbursement does not apply to losses covered under Section 4.5 (Roadside Assistance).

## 4.4 Exclusions
The following are excluded from coverage under this section:
- Damage from racing or speed contests
- Damage while the vehicle is used for commercial delivery services
- Damage occurring outside the policy territory
- Wear, tear, freezing, or mechanical breakdown

## 4.5 Roadside Assistance
The Company will reimburse reasonable towing and labor costs up to $100 per disablement,
limited to four disablements per policy period. Roadside assistance is a separate benefit
from rental reimbursement and does not require a collision or comprehensive claim.
"""

print(f"Document length: {len(sample_document)} characters, "
      f"{len(sample_document.split())} words, "
      f"{sample_document.count(chr(10)+chr(10))} paragraph breaks.")


---
## Strategy 1 — Fixed-Size Chunking (2:00–5:00)

The simplest possible approach: cut the text every *N* characters, no matter what's there.
Watch what happens to clause boundaries — this is the failure mode Module 1 warned about.


In [ ]:
def fixed_size_chunks(text, chunk_size=200):
    return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

chunks_fixed = fixed_size_chunks(sample_document, chunk_size=200)

print(f"Produced {len(chunks_fixed)} chunks\n")
for i, c in enumerate(chunks_fixed[:4]):
    print(f"--- Chunk {i+1} ({len(c)} chars) ---")
    print(c)
    print()


**Discussion point:** look at chunk 2 or 3 — a dollar amount or a clause is almost certainly
sliced in half. If a user asks "what's the deductible for collision coverage?", the retrieved
chunk may not contain the number at all. This is fixed-size chunking's core failure mode.


---
## Strategy 2 — Sentence-Based Chunking (5:00–8:00)

Instead of cutting blindly, split on sentence boundaries and group a few sentences per
chunk. Better — no more mid-sentence cuts — but still ignores document structure (a chunk
can straddle two unrelated sections).


In [ ]:
import re

def sentence_chunks(text, sentences_per_chunk=3):
    # Naive sentence splitter (good enough for a demo; use a real NLP library in production)
    clean = re.sub(r"\n+", " ", text).strip()
    sentences = re.split(r"(?<=[.!?])\s+", clean)
    sentences = [s.strip() for s in sentences if s.strip()]
    return [" ".join(sentences[i:i + sentences_per_chunk])
            for i in range(0, len(sentences), sentences_per_chunk)]

chunks_sentence = sentence_chunks(sample_document, sentences_per_chunk=3)

print(f"Produced {len(chunks_sentence)} chunks\n")
for i, c in enumerate(chunks_sentence[:4]):
    print(f"--- Chunk {i+1} ({len(c)} chars) ---")
    print(c)
    print()


**Discussion point:** every sentence is now intact, but notice a chunk can still mix the
tail of one clause (e.g., Collision) with the start of the next (Comprehensive) — headings
and section identity are lost entirely.


---
## Strategy 3 — Recursive Character Chunking with Overlap (8:00–11:00)

The production-grade default for unstructured text: try to split on paragraph breaks first,
fall back to sentences, then words, only if a chunk is still too big. Add a small **overlap**
between consecutive chunks so context isn't lost at the boundary.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks_recursive = splitter.split_text(sample_document)

print(f"Produced {len(chunks_recursive)} chunks\n")
for i, c in enumerate(chunks_recursive):
    print(f"--- Chunk {i+1} ({len(c)} chars) ---")
    print(c)
    print()


**Discussion point:** compare the chunk boundaries here to Strategy 1. The 50-character
overlap means a fact right at a boundary (e.g., "$500 deductible") is very likely to appear
fully in at least one chunk. This is the default building block most production RAG systems
start from — but it still doesn't know that "4.1" and "4.2" are different legal sections.


---
## Strategy 4 — Structure-Aware Chunking (by Heading) (11:00–14:00)

This is the pattern from the Unit 3 deck's "Ingestion & Chunking Strategy" slide: split by
the document's own structure (Markdown headings, in this case) so every chunk is a complete,
coherent clause — and tag each chunk with metadata (its section) for filtering and citation.


In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [("##", "section")]
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
structure_chunks = md_splitter.split_text(sample_document)

print(f"Produced {len(structure_chunks)} chunks\n")
for i, doc in enumerate(structure_chunks):
    print(f"--- Chunk {i+1} | metadata: {doc.metadata} ({len(doc.page_content)} chars) ---")
    print(doc.page_content.strip())
    print()


**Discussion point:** every chunk is now exactly one policy clause, and carries a `section`
tag as metadata — exactly what lets a production system filter to "Section 4.4 only" or cite
"per Section 4.3" in a generated answer. This is usually the best starting point for
enterprise documents that already have real structure (headings, numbered clauses, tables).


---
## Strategy 5 — Semantic Chunking (14:00–17:00)

For documents *without* clean structure (transcripts, freeform notes), an alternative is to
split wherever the **meaning** shifts: embed each sentence, measure similarity between
consecutive sentences, and cut where similarity drops sharply.

This cell uses a real embedding model — it needs internet access to download the model the
first time it runs (works in Colab; may be blocked in a locked-down sandbox).


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import re

embedder = SentenceTransformer("all-MiniLM-L6-v2")

def semantic_chunks(text, similarity_drop_threshold=0.25):
    clean = re.sub(r"\n+", " ", text).strip()
    clean = re.sub(r"#+\s*", "", clean)  # strip markdown heading markers for this demo
    sentences = re.split(r"(?<=[.!?])\s+", clean)
    sentences = [s.strip() for s in sentences if s.strip()]

    embeddings = embedder.encode(sentences, normalize_embeddings=True)

    # cosine similarity between each sentence and the next
    sims = [float(np.dot(embeddings[i], embeddings[i + 1])) for i in range(len(embeddings) - 1)]

    chunks, current = [], [sentences[0]]
    for i, sim in enumerate(sims):
        if sim < (1 - similarity_drop_threshold):
            chunks.append(" ".join(current))
            current = [sentences[i + 1]]
        else:
            current.append(sentences[i + 1])
    chunks.append(" ".join(current))
    return chunks, sims

chunks_semantic, similarities = semantic_chunks(sample_document)

print(f"Produced {len(chunks_semantic)} chunks\n")
for i, c in enumerate(chunks_semantic):
    print(f"--- Chunk {i+1} ({len(c)} chars) ---")
    print(c)
    print()

print("Sentence-to-sentence similarity scores (a big drop = a chunk boundary):")
print([round(s, 2) for s in similarities])


**Discussion point:** semantic chunking needs no heading structure at all — it discovers
topic shifts (Collision → Comprehensive → Rental → Exclusions) purely from meaning. It's
more expensive to compute (an embedding call per sentence) and the threshold needs tuning,
which is why structure-aware chunking (Strategy 4) is usually preferred *when structure
exists* — semantic chunking is the fallback for when it doesn't.


---
## Side-by-Side Comparison (17:00–19:00)

A quick numeric summary of everything above — chunk count and size distribution are the
first things to check when evaluating a chunking strategy for a real corpus.


In [ ]:
import statistics

strategies = {
    "Fixed-Size (200 chars)": chunks_fixed,
    "Sentence-Based (3/chunk)": chunks_sentence,
    "Recursive + Overlap": chunks_recursive,
    "Structure-Aware (by heading)": [d.page_content for d in structure_chunks],
    "Semantic": chunks_semantic,
}

print(f"{'Strategy':<32}{'# Chunks':<10}{'Avg Len':<10}{'Min Len':<10}{'Max Len':<10}")
for name, chunks in strategies.items():
    lengths = [len(c) for c in chunks]
    print(f"{name:<32}{len(chunks):<10}{int(statistics.mean(lengths)):<10}{min(lengths):<10}{max(lengths):<10}")


In [ ]:
# Optional: quick visualization of chunk-size distribution
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
for name, chunks in strategies.items():
    lengths = [len(c) for c in chunks]
    ax.scatter([name] * len(lengths), lengths, alpha=0.7, s=60)

ax.set_ylabel("Chunk length (characters)")
ax.set_title("Chunk size distribution by strategy")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


---
## Wrap-Up (19:00–20:00)

**What we just watched, in one line each:**
- **Fixed-size** — fastest to write, most likely to cut a fact in half.
- **Sentence-based** — no more mid-sentence cuts, still blind to document structure.
- **Recursive + overlap** — solid general-purpose default; overlap protects boundary facts.
- **Structure-aware** — best choice whenever real structure (headings, clauses) exists; also gives you free metadata for citations and filtering.
- **Semantic** — best fallback for structureless text; costs more to compute.

**Rule of thumb from Module 1:** prefer structure-aware chunking for enterprise documents
that already have headings or numbered clauses (contracts, policies, filings); fall back to
semantic or recursive chunking for freeform text (call transcripts, chat logs, notes).

The extension tasks below are meant to be done **after** this live session, at your own pace.


---
---
# Extension Tasks

Pick any of these — they build directly on the code above. None require new API keys beyond
what you already set up for Unit 3's other demos, except where noted.

## Task 1 — Tune Chunk Size and Overlap (Easy)
Re-run Strategy 3 (Recursive) with `chunk_size` values of 100, 300, and 600, and `chunk_overlap`
values of 0, 50, and 150. For each combination, note the chunk count and skim 2–3 chunks.
At what size do clauses start getting cut again? At what overlap does redundancy start to
feel wasteful?

## Task 2 — Build a Structure-Aware Splitter for a Different Document Type (Medium)
The Markdown splitter in Strategy 4 works because our sample document already uses `##`
headings. Take a **numbered-clause** document instead (e.g., paste in a real contract or
write one with clauses like "Section 1.", "Section 2.") and write a splitter using a regex
on the clause numbering pattern instead of Markdown headers. Compare its output to what
Strategy 3 (Recursive) produces on the same text.

## Task 3 — Measure Retrieval Quality, Not Just Chunk Shape (Medium–Hard)
This is the real test of a chunking strategy. Using the RAG pipeline from **Demo 1**
(`Demo1_End_to_End_RAG_Pipeline.ipynb`):
1. Re-index the same knowledge base using two different chunking strategies from this notebook.
2. Run the same 4–5 test queries against both indexes.
3. Compare: does the *right* chunk get retrieved in the top-3 for each strategy? Does the
   faithfulness check from Demo 1 pass more often with one strategy than the other?

## Task 4 — Tune the Semantic Chunking Threshold (Medium)
In Strategy 5, try `similarity_drop_threshold` values of 0.1, 0.25, and 0.4. Plot how the
number of chunks changes. Too low a threshold merges everything into one chunk; too high
splits every sentence into its own chunk. Find the value that best matches the five clauses
in the sample document.

## Task 5 — Chunk a Real PDF (Hard)
Swap `sample_document` for text extracted from a real PDF (e.g., a public 10-K filing or an
insurance policy you have permission to use). You'll need a PDF text extraction step first
(e.g., `pypdf` or `pdfplumber`). Real documents have messier structure than this demo's
sample — expect to combine structure-aware splitting with a recursive fallback for sections
that don't parse cleanly.

## Task 6 — Cost & Latency Back-of-Envelope (Easy)
For a corpus of 10,000 pages, estimate the number of chunks each strategy would produce
(using the average chunk-length numbers from the comparison table above), and roughly how
many embedding API calls that implies. Which strategy is cheapest to index? Which is
cheapest to *re-index* after a single document changes?
